<a href="https://colab.research.google.com/github/rudrakxh11/LanguageTransalator-ChatGPT-Model/blob/main/ChatGPT_model_for_Language_Transaltion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Gathering Data

In [31]:
import tensorflow as tf
extracted_data_dir = tf.keras.utils.get_file(
    fname='fra-eng.zip',
    origin='http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip',
    extract=True
)

In [32]:
import pathlib #help extracting the file
text_file = pathlib.Path(extracted_data_dir) / 'fra.txt'
#file handling, got the file in english french pair so read it in pair wise

In [25]:
print(text_file)

/root/.keras/datasets/fra-eng_extracted/fra.txt


In [26]:
with open(text_file) as fp:
  text_pair=[line for line in fp]


In [29]:
import random
for i in range(5):
  print(random.choice(text_pair))

I would do anything to get a job.	Je ferais n'importe quoi pour obtenir un emploi.

Hunger is the best sauce.	La faim est le meilleur des cuisiniers.

I am from Brazil.	Je viens du Brésil.

I like listening to music.	J'aime écouter de la musique.

He drank a cup of coffee.	Il a bu une tasse de café.



In [39]:
#convert to unicode
import unicodedata
import re

#remove all language and convert all character to lower case so all the word having similar meaning can be kept in a single way itself
def normalize(line):
  line=unicodedata.normalize("NFKC", line.strip().lower())
  line=re.sub(r"^([^ \w])(?!\\s)",r"\1", line) #doing this for speacial characters...so that they can be identified easily #if any kind of character along the line...we're putting some spaces over there
  line=re.sub(r"(\s[^ \w])(?!\\s)",r"\1", line) #if any kind of character along the last of the line...we're putting some spaces over there
  line=re.sub(r"((?!\\s)[^ \w])$",r"\1", line)
  line=re.sub(r"((?!\\s)[^ \w])\\s",r"\1", line)

  parts = line.split("\t")
  if len(parts) != 2:
    return None # Return None for malformed lines

  eng, fre = parts
  fre = '[start]' + fre + '[end]'
  return eng,fre

In [43]:
with open(text_file) as fp:
  # Filter out lines for which normalize returns None
  text_pairs=[normalized_line for line in fp if (normalized_line := normalize(line)) is not None]

In [44]:
for i in range(5):
  print(random.choice(test_pairs))

('i give you everything you ask for, but you never seem satisfied.', '[start]je te donne tout ce que tu demandes mais tu ne sembles jamais satisfait.[end]')
('i need some answers.', '[start]il me faut des réponses.[end]')
('push the button, please.', '[start]veuillez appuyer sur le bouton.[end]')
("don't spread yourself too thin.", '[start]ne fais pas trop de choses à la fois.[end]')
('he may come and see us tonight.', "[start]il se peut qu'il vienne nous voir ce soir.[end]")


In [48]:
eng_toke,fre_toke=set(),set()
eng_maxlen, fre_maxlen=0,0
for eng,fre in text_pairs:
  # Use temporary variables for split results to avoid overwriting the sets
  current_eng_tokens = eng.split()
  current_fre_tokens = fre.split()

  eng_maxlen=max(eng_maxlen, len(current_eng_tokens))
  fre_maxlen=max(fre_maxlen, len(current_fre_tokens))

  # Update the original set variables
  eng_toke.update(current_eng_tokens)
  fre_toke.update(current_fre_tokens)

print(len(eng_toke))
print(len(fre_toke))
print(eng_maxlen)
print(fre_maxlen)

25365
44581
47
54


In [49]:
import pickle
with open("text_pairs.pickle",'wb') as fp:
  pickle.dump(text_pairs,fp)


transformer creation